# Phase 3: the junior + senior (AI agent) experiment

- **Junior:** a Random Forest answers every exam row and says how sure it is.
- **Senior:** Gemini looks only at the rows where the junior is less than 70% sure, about 470 rows over both GPUs.
- The senior is tested in 4 ways: code only; code + fact sheet; code + all three tools at once; and as an **agent** that chooses its own tools.

Exam: new applications (the fair exam), same piles as the fine-tuned models.

**Before you start:** your Gemini key must be saved in Colab **Secrets** as `GEMINI_API_KEY`, with **Notebook access** switched on.

**How to run:**
1. **No GPU needed:** **Runtime → Change runtime type → CPU** (the AI runs on Google's servers, so save your GPU time).
2. **Runtime → Run all.** Allow Google Drive access (tick **Select all**). If Colab asks whether this notebook may read the secret `GEMINI_API_KEY`, click **Grant access**.
3. The "Connection check" cell must print `Connection OK`. If it prints an error, send Claude a screenshot.
4. The full run is roughly 2,800 AI questions with a short pause between them (free-tier friendly), so expect **several hours**. You can leave it running.
5. **If it stops** (free daily limit, or Colab disconnects): nothing is lost. Every answer is saved to `devmap_results/phase3` in your Drive immediately. Run the notebook again later (limits reset daily) and it continues where it stopped.

In [ ]:
import os, shutil, subprocess
subprocess.run(["pip", "install", "-q", "google-genai"], check=True)
if not os.path.exists("GM_SOURCE_CODE"):
    subprocess.run(["git", "clone", "--depth", "1", "-q",
                    "https://github.com/marco-siino/GM_SOURCE_CODE.git"], check=True)
os.makedirs("data", exist_ok=True)
shutil.copy("GM_SOURCE_CODE/AMD_DS/dataset-devmap-amd.csv", "data")
shutil.copy("GM_SOURCE_CODE/NVIDIA_DS/dataset-devmap-nvidia.csv", "data")
print("Ready.")

## Helper code (no need to read or change)

In [ ]:
%%writefile devmap.py
"""
Shared helpers: load the DevMap dataset and build the four "exams".

Words used here (same as in our chats):
  row      = one line of the dataset (code + two job numbers + answer)
  program  = one piece of OpenCL code (a "kernel"); several rows can share it
  study    = the rows a model learns from   (training set)
  exam     = the rows it is scored on       (test set)

  application = the benchmark a program was taken from (e.g. "npb-3.3-BT");
                one application contains several programs. 71 in total.
  collection  = the benchmark suite an application belongs to (NPB, Rodinia,
                ...). 7 in total.

The four exams, from easiest to hardest:
  old             - the base paper's exam: shuffle the ROWS, cut into 5 piles.
                    Rows of the same program can land on both sides.
  unseen_program  - shuffle the PROGRAMS, cut into 5 piles. Every row of a
                    program is on one side only.
  unseen_app      - shuffle the APPLICATIONS, cut into 5 piles. Needed because
                    programs of one application share the same job numbers,
                    so those numbers give the application away.
  unseen_suite    - hold out one whole collection at a time, 7 rounds.
"""
import os
import re

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

DATA_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "data")
FILES = {"amd": "dataset-devmap-amd.csv", "nvidia": "dataset-devmap-nvidia.csv"}

# The hand-counted features that ship with the dataset (Grewe et al.):
#   comp      = number of compute operations in the code
#   rational  = number of relational (comparison) operations
#   mem       = number of memory accesses
#   localmem  = number of accesses to local memory
#   coalesced = number of memory accesses that are neatly side by side
#   atomic    = number of atomic operations (always 0 here)
#   transfer  = how much data is sent to the device  (job number 1)
#   wgsize    = how the work is batched               (job number 2)
STATIC_FEATURES = ["comp", "rational", "mem", "localmem", "coalesced", "atomic"]
JOB_FEATURES = ["transfer", "wgsize"]

SUITES = ["amd-sdk", "npb", "nvidia-sdk", "parboil", "polybench", "rodinia", "shoc"]


def suite_of(benchmark_name):
    """Which collection a program comes from, read off its name."""
    b = benchmark_name.lower()
    if b.startswith("amd"):
        return "amd-sdk"
    if b.startswith("npb"):
        return "npb"
    if b.startswith("nvidia"):
        return "nvidia-sdk"
    if b.startswith("parboil"):
        return "parboil"
    if b.startswith("polybench"):
        return "polybench"
    if b.startswith("rodinia"):
        return "rodinia"
    if b.startswith("shoc"):
        return "shoc"
    raise ValueError("unknown collection for " + benchmark_name)


def app_of(benchmark_name):
    """'npb-3.3-BT-exact_rhs1' -> 'npb-3.3-BT' (collection, version, application)."""
    m = re.match(r"^(.*?-\d+(?:\.\d+)*)-([^-]+)-(.*)$", benchmark_name)
    if not m:
        raise ValueError("cannot read application from " + benchmark_name)
    return m.group(1) + "-" + m.group(2)


def code_key(src):
    """Two rows are 'the same program' when their code text is identical
    once extra spaces and line breaks are ignored."""
    return re.sub(r"\s+", " ", str(src)).strip()


def load(machine, data_dir=DATA_DIR):
    """Load one answer sheet ('amd' or 'nvidia') with a few helper columns."""
    df = pd.read_csv(os.path.join(data_dir, FILES[machine]), sep=",")
    df = df.loc[:, [c for c in df.columns if not c.startswith("Unnamed")]].copy()
    df["row_id"] = np.arange(len(df))           # stable id = original file order
    df["suite"] = df["benchmark"].map(suite_of)
    df["app"] = df["benchmark"].map(app_of)
    keys = df["src"].map(code_key)
    df["program_id"] = pd.factorize(keys)[0]     # same code text -> same id
    df["label"] = (df["oracle"] == "GPU").astype(int)   # 0 = CPU, 1 = GPU
    return df


# ----------------------------------------------------------------------------
# The four exams. Each returns a list of rounds; a round is a dict with
# 'name', 'study' (row_ids) and 'exam' (row_ids).
# ----------------------------------------------------------------------------
def old_exam(df):
    """Exactly what the base paper's notebooks do:
    df.sample(frac=1, random_state=1) and then KFold(5, shuffle=False)."""
    shuffled = df.sample(frac=1, random_state=1).reset_index(drop=True)
    rounds = []
    for i, (tr, te) in enumerate(KFold(n_splits=5, shuffle=False).split(shuffled)):
        rounds.append({
            "name": f"pile{i + 1}",
            "study": shuffled.iloc[tr]["row_id"].tolist(),
            "exam": shuffled.iloc[te]["row_id"].tolist(),
        })
    return rounds


def group_piles(groups, labels, n_piles=5, seed=1):
    """Deal whole groups (programs or applications) into n_piles piles.

    Our own version instead of scikit-learn's StratifiedGroupKFold, because
    that one gives different piles in different library versions (Colab's
    1.6 put 364 rows in one AMD pile and 15 in another). This uses only
    numpy's fixed random generator, so the piles are the same everywhere.

    How it deals: shuffle the groups (seed), then take them biggest first and
    put each one in the pile where it keeps the piles most even, both in
    size and in their share of GPU answers. Returns one pile number per row."""
    groups = np.asarray(groups)
    labels = np.asarray(labels)
    names, inverse = np.unique(groups, return_inverse=True)
    sizes = np.bincount(inverse)
    gpus = np.bincount(inverse, weights=labels)
    order = np.random.RandomState(seed).permutation(len(names))
    order = order[np.argsort(-sizes[order], kind="stable")]       # biggest first, ties in shuffled order
    target_rows = len(groups) / n_piles
    target_gpu = max(labels.sum() / n_piles, 1.0)
    pile_rows = np.zeros(n_piles)
    pile_gpu = np.zeros(n_piles)
    pile_of_group = np.empty(len(names), dtype=int)
    for g in order:
        costs = []
        for p in range(n_piles):
            rows, gpu = pile_rows.copy(), pile_gpu.copy()
            rows[p] += sizes[g]
            gpu[p] += gpus[g]
            costs.append((((rows - target_rows) / target_rows) ** 2).sum()
                         + (((gpu - target_gpu) / target_gpu) ** 2).sum())
        best = int(np.argmin(costs))                               # ties -> lowest pile number
        pile_of_group[g] = best
        pile_rows[best] += sizes[g]
        pile_gpu[best] += gpus[g]
    return pile_of_group[inverse]


def fair_exam_unseen_programs(df, seed=1):
    """5 rounds; all rows of a program stay on the same side. Piles are also
    kept roughly balanced in size and CPU/GPU answers."""
    pile = group_piles(df["program_id"], df["label"], 5, seed)
    rounds = []
    for i in range(5):
        tr, te = np.where(pile != i)[0], np.where(pile == i)[0]
        rounds.append({
            "name": f"pile{i + 1}",
            "study": df.iloc[tr]["row_id"].tolist(),
            "exam": df.iloc[te]["row_id"].tolist(),
        })
    return rounds


def fair_exam_unseen_apps(df, seed=1):
    """5 rounds; all programs of an application stay on the same side. If the
    exact same code also exists under another application, those study rows
    are dropped so the exam code is truly unseen."""
    pile = group_piles(df["app"], df["label"], 5, seed)
    rounds = []
    for i in range(5):
        tr, te = np.where(pile != i)[0], np.where(pile == i)[0]
        exam = df.iloc[te]
        study = df.iloc[tr]
        study = study[~study["program_id"].isin(exam["program_id"])]
        rounds.append({
            "name": f"pile{i + 1}",
            "study": study["row_id"].tolist(),
            "exam": exam["row_id"].tolist(),
        })
    return rounds


def fair_exam_unseen_collection(df):
    """7 rounds; each round holds out one whole collection. If a held-out
    program's exact code also exists in another collection, those study rows
    are dropped so the exam code is truly unseen."""
    rounds = []
    for suite in SUITES:
        exam = df[df["suite"] == suite]
        study = df[(df["suite"] != suite) & (~df["program_id"].isin(exam["program_id"]))]
        rounds.append({
            "name": suite,
            "study": study["row_id"].tolist(),
            "exam": exam["row_id"].tolist(),
        })
    return rounds


def build_exam(df, exam, seed=1):
    if exam == "old":
        return old_exam(df)
    if exam == "unseen_program":
        return fair_exam_unseen_programs(df, seed)
    if exam == "unseen_app":
        return fair_exam_unseen_apps(df, seed)
    if exam == "unseen_suite":
        return fair_exam_unseen_collection(df)
    raise ValueError(exam)


EXAMS = ["old", "unseen_program", "unseen_app", "unseen_suite"]
RESHUFFLED_EXAMS = ["unseen_program", "unseen_app"]   # these change with the shuffle seed


def leak_report(df, rounds):
    """For each round: how many exam rows have code that is also in the
    study rows (seen) vs. not (new)."""
    pid = df.set_index("row_id")["program_id"]
    out = []
    for r in rounds:
        study_programs = set(pid.loc[r["study"]])
        seen = sum(1 for x in r["exam"] if pid.loc[x] in study_programs)
        out.append({"round": r["name"], "study_rows": len(r["study"]),
                    "exam_rows": len(r["exam"]), "exam_seen_code": seen,
                    "exam_new_code": len(r["exam"]) - seen})
    return pd.DataFrame(out)

In [ ]:
%%writefile embed_lib.py
"""
Turns each distinct piece of DevMap code into a list of numbers (an
"embedding") using a pretrained model, WITHOUT any training. These numbers
are the model's own summary of what the code looks like and does.

Only the code text goes in - never the job numbers - so the embedding cannot
act as a name tag for the job size.

  embed(...)            runs a model over the 249 distinct kernels, saves an .npz
  load_embeddings(path) reads it back as {code id: vector}
  code_ids(df)          the code id of every dataset row (to look vectors up)

A code id is the MD5 of the whitespace-normalised code (devmap.code_key), so
the same code always gets the same id on any computer.

Two ways of reading a kernel:
  cut      - the first max_len tokens only (what the fine-tuned models do;
             87 of the 249 kernels are longer than 512 tokens and get cut)
  pieces   - the whole kernel, split into pieces of at most max_len tokens;
             the kernel's vector is the average of the pieces' vectors,
             weighted by how many tokens each piece has
"""
import hashlib
import os

import numpy as np
import torch

import devmap

# Each model: where to download it, how many tokens one piece may have, how
# to read long kernels, and model-specific loading details from its model card.
SPECS = {
    # the two models fine-tuned in the main study (first 512 tokens only)
    "codeberta": {"hub": "huggingface/CodeBERTa-language-id", "max_len": 512, "pieces": False},
    "graphcodebert": {"hub": "microsoft/graphcodebert-base", "max_len": 512, "pieces": False},
    # same model, but reading the whole kernel in pieces
    "graphcodebert-pieces": {"hub": "microsoft/graphcodebert-base", "max_len": 512, "pieces": True},
    # newer models (Phase 1)
    "unixcoder": {"hub": "microsoft/unixcoder-base", "max_len": 1024, "pieces": True},
    "codet5p": {"hub": "Salesforce/codet5p-110m-embedding", "max_len": 512, "pieces": True,
                "remote": True},
    "codesage": {"hub": "codesage/codesage-base-v2", "max_len": 1024, "pieces": True,
                 "remote": True, "tokenizer_args": {"add_eos_token": True}},
    "jina-code": {"hub": "jinaai/jina-embeddings-v2-base-code", "max_len": 4096, "pieces": True,
                  "remote": True},
    "codebert": {"hub": "microsoft/codebert-base", "max_len": 512, "pieces": False},
}
MODELS = {k: v["hub"] for k, v in SPECS.items()}       # kept for older scripts


def code_id(src):
    return hashlib.md5(devmap.code_key(src).encode("utf-8")).hexdigest()


def code_ids(df):
    return df["src"].map(code_id).to_numpy()


def unique_codes(data_dir=devmap.DATA_DIR):
    """(ids, texts) for every distinct kernel. The code is the same on both
    machines' answer sheets, so the AMD file is enough."""
    df = devmap.load("amd", data_dir)
    seen, ids, texts = set(), [], []
    for src in df["src"]:
        i = code_id(src)
        if i not in seen:
            seen.add(i)
            ids.append(i)
            texts.append(devmap.code_key(src))
    return ids, texts


def compat_shims():
    """Some models ship their own code written for older versions of the
    transformers library, which import helpers that later versions moved or
    removed. Put those helpers back where the old code looks for them. This only
    makes the old code importable; it does not change any computation."""
    import transformers.modeling_utils as mu
    import transformers.pytorch_utils as pu
    if not hasattr(mu, "Conv1D") and hasattr(pu, "Conv1D"):          # CodeSage
        mu.Conv1D = pu.Conv1D
    if not hasattr(pu, "find_pruneable_heads_and_indices"):          # jina-code on transformers 5
        def find_pruneable_heads_and_indices(heads, n_heads, head_size, already_pruned_heads):
            mask = torch.ones(n_heads, head_size)
            heads = set(heads) - already_pruned_heads
            for head in heads:
                head = head - sum(1 if h < head else 0 for h in already_pruned_heads)
                mask[head] = 0
            mask = mask.view(-1).contiguous().eq(1)
            index = torch.arange(len(mask))[mask].long()
            return heads, index
        pu.find_pruneable_heads_and_indices = find_pruneable_heads_and_indices


def n_tokens(tokenizer, text):
    return len(tokenizer([text], truncation=False)["input_ids"][0])


def split_into_pieces(text, tokenizer, max_len, step=24):
    """Split code into consecutive pieces of at most max_len tokens (special
    tokens included), cutting only between words. Every word lands in exactly
    one piece, in order."""
    words = text.split(" ")
    pieces, current = [], []
    for i in range(0, len(words), step):
        block = words[i:i + step]
        trial = current + block
        if current and n_tokens(tokenizer, " ".join(trial)) > max_len:
            pieces.append(" ".join(current))
            current = list(block)
        else:
            current = trial
    if current:
        pieces.append(" ".join(current))
    return pieces


def _vectors(model, tokenizer, text, max_len, device):
    """(mean vector, first-token vector, tokens used) for one piece of text."""
    enc = tokenizer([text], truncation=True, max_length=max_len, return_tensors="pt")
    ids = enc["input_ids"].to(device)
    mask = enc["attention_mask"].to(device) if "attention_mask" in enc else torch.ones_like(ids)
    try:
        out = model(input_ids=ids, attention_mask=mask)
    except TypeError:                                   # some remote models take ids only
        out = model(ids)
    first = out[0] if not torch.is_tensor(out) else out
    first = first.float()
    if first.dim() == 3:                                # token vectors -> average over real tokens
        m = mask.unsqueeze(-1).float()
        mean = (first * m).sum(1) / m.sum(1).clamp(min=1)
        cls = first[:, 0]
    else:                                               # the model already returns one vector
        mean = cls = first.reshape(1, -1)
    return mean[0].cpu().numpy(), cls[0].cpu().numpy(), int(mask.sum().item())


def embed(model_key, data_dir, out_dir, tokenizer=None, make_encoder=None, device=None,
          max_len=None, pieces=None, log=print):
    """Saves <out_dir>/embeddings__<model_key>.npz with, per kernel:
       mean   - average of the model's token vectors (main choice)
       cls    - the first token's vector
       tokens - tokens actually read;  n_pieces - pieces read
    Skips the work if the file already exists."""
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, f"embeddings__{model_key}.npz")
    if os.path.exists(path):
        log(f"{path} already exists, skipping")
        return path
    spec = SPECS.get(model_key, {})
    max_len = max_len or spec.get("max_len", 512)
    pieces = spec.get("pieces", False) if pieces is None else pieces
    if tokenizer is None or make_encoder is None:
        from transformers import AutoModel, AutoTokenizer
        compat_shims()
        hub, remote = spec["hub"], spec.get("remote", False)
        tokenizer = AutoTokenizer.from_pretrained(hub, trust_remote_code=remote, **spec.get("tokenizer_args", {}))
        make_encoder = lambda: AutoModel.from_pretrained(hub, trust_remote_code=remote)
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model = make_encoder().to(device).eval()

    ids, texts = unique_codes(data_dir)
    means, clss, tokens, n_pieces = [], [], [], []
    with torch.no_grad():
        for k, text in enumerate(texts):
            # On a small GPU a very long piece may not fit in memory. Then read that
            # kernel again in smaller pieces (halving until it fits), rather than crash.
            limit = max_len
            while True:
                try:
                    parts = split_into_pieces(text, tokenizer, limit) if (pieces or limit < max_len) else [text]
                    vecs = [_vectors(model, tokenizer, p, limit, device) for p in parts]
                    break
                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache()
                    if limit <= 128:
                        raise
                    limit //= 2
                    log(f"   kernel {k + 1}: GPU memory full, reading it in pieces of {limit} tokens instead")
            w = np.array([v[2] for v in vecs], dtype=float)
            means.append(np.average(np.stack([v[0] for v in vecs]), axis=0, weights=w))
            clss.append(vecs[0][1])
            tokens.append(int(w.sum()))
            n_pieces.append(len(parts))
            if (k + 1) % 25 == 0 or k + 1 == len(texts):
                log(f"   {model_key}: {k + 1}/{len(texts)} kernels")
    tokens, n_pieces = np.array(tokens), np.array(n_pieces)
    np.savez_compressed(path, ids=np.array(ids), mean=np.stack(means), cls=np.stack(clss),
                        tokens=tokens, n_pieces=n_pieces, model=spec.get("hub", model_key))
    log(f"saved {path}: {len(ids)} kernels, vectors of {len(means[0])} numbers, "
        f"{int((n_pieces > 1).sum())} kernels read in more than one piece")
    return path


def load_embeddings(path, pooling="mean"):
    z = np.load(path, allow_pickle=False)
    return dict(zip(z["ids"].tolist(), z[pooling]))

In [ ]:
%%writefile features.py
"""
Hand-made features shared by the name-tag test and the hybrid model.

  counts  - the six code counts that ship with DevMap (comp, rational, mem,
            localmem, coalesced, atomic)
  raw job numbers - transfer and wgsize exactly as stored. Nearly unique per
            application, so they can act as a "name tag".
  ratios  - scale-free versions that keep the useful information (how much
            data per operation, how much work per work-item) without being a
            name tag on their own:
              transfer_per_op = transfer / (comp + mem)   data moved per operation
              coalesced_share = coalesced / mem           share of tidy memory accesses
              local_share     = localmem / mem            share of local-memory accesses
              comp_per_mem    = comp / mem                arithmetic intensity
              ops_per_item    = (comp + mem) / wgsize     work per work-item in a batch
  grewe4  - the four features Grewe et al. (2013) designed. F3 still multiplies
            by the raw wgsize.
"""
import numpy as np
import pandas as pd

import devmap


def _safe(a, b):
    return (a / b.replace(0, np.nan)).fillna(0.0)


def counts(df):
    return df[devmap.STATIC_FEATURES].astype(float)


def job_numbers(df):
    return df[devmap.JOB_FEATURES].astype(float)


def ratios(df):
    X = df[devmap.STATIC_FEATURES + devmap.JOB_FEATURES].astype(float)
    ops = X["comp"] + X["mem"]
    return pd.DataFrame({
        "transfer_per_op": _safe(X["transfer"], ops),
        "coalesced_share": _safe(X["coalesced"], X["mem"]),
        "local_share": _safe(X["localmem"], X["mem"]),
        "comp_per_mem": _safe(X["comp"], X["mem"]),
        "ops_per_item": _safe(ops, X["wgsize"]),
    }, index=df.index)


def grewe4(df):
    X = df[devmap.STATIC_FEATURES + devmap.JOB_FEATURES].astype(float)
    r = ratios(df)
    return pd.DataFrame({
        "F1": r["transfer_per_op"], "F2": r["coalesced_share"],
        "F3": r["local_share"] * X["wgsize"], "F4": r["comp_per_mem"],
    }, index=df.index)


def forest(df):
    """Our original Random Forest baseline: counts + raw job numbers + Grewe's four."""
    return pd.concat([counts(df), job_numbers(df), grewe4(df)], axis=1)

In [ ]:
%%writefile agent_lib.py
"""
Phase 3: a "junior + senior" device-mapping system.

  junior  - a Random Forest on the hand-counted features (fast, cheap). It answers
            every exam row and says how sure it is.
  senior  - a large language model (LLM). It only looks at the rows where the
            junior is less than THRESHOLD sure, and may use three tools:
              fact_sheet      the kernel's counts and job sizes in plain words,
                              plus derived ratios (work per memory access, ...)
              junior_opinion  the Random Forest's answer and how sure it is
              similar_kernels the 3 most similar kernels from the STUDY rows
                              (never the exam application), with their measured
                              answers at the nearest job size
The senior never runs anything on hardware.

Four senior variants, all scored on the same hard rows:
  code        the kernel code and job sizes only (no tools)
  facts       + the fact sheet
  all_tools   + fact sheet + junior opinion + similar kernels, all given at once
  agent       the LLM decides which tools to call (up to 3), one at a time, then answers

Final system answer = junior on rows it is sure about + senior on the hard rows.
If the senior fails (bad reply, error), the junior's answer is kept and the
failure is recorded.

Every LLM reply is saved to a CSV as soon as it arrives, so an interrupted run
(Colab disconnect, free-tier daily limit) resumes where it stopped.
"""
import csv
import json
import os
import re
import time

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

import devmap
import embed_lib
import features as F

THRESHOLD = 0.7           # junior hands a row to the senior when max(p, 1-p) < THRESHOLD
JUNIOR_SEED = 1
EXAM, SPLIT_SEED = "unseen_app", 1
VARIANTS = ["all_tools", "agent", "facts", "code"]     # run order: most important first
MAX_TOOL_CALLS = 3
CODE_CHARS = 6000          # longer kernels are shown truncated (about 2,000 tokens)

PLATFORMS = {
    "amd": "CPU: Intel Core i7-3820 (4 cores, 3.6 GHz). GPU: AMD Radeon HD 7970 'Tahiti' (discrete, PCIe).",
    "nvidia": "CPU: Intel Core i7-3820 (4 cores, 3.6 GHz). GPU: NVIDIA GeForce GTX 970 (discrete, PCIe).",
}

TASK = """You decide whether an OpenCL kernel runs faster on the CPU or on the GPU of this machine.
{platform}
Each case is one kernel launched with a given job size:
- transfer: bytes copied between CPU memory and GPU memory for this launch
- work-group size: number of work-items per OpenCL work-group
The correct answer was measured: whichever device actually ran this launch faster.
Typical considerations: data-transfer cost over PCIe, amount of parallel work, arithmetic per
memory access, coalesced versus scattered memory access, local-memory use, branching."""

ANSWER_FORMAT = """Reply with ONE JSON object and nothing else:
{"device": "CPU" or "GPU", "confidence": number between 0 and 1, "reason": "at most two sentences citing the evidence you used"}"""

AGENT_FORMAT = """You may use tools before answering. Each tool takes no arguments and is about THIS case:
- fact_sheet: the kernel's operation counts, job sizes and derived ratios in plain words
- junior_opinion: what a Random Forest trained on measured cases predicts, and how sure it is
- similar_kernels: the 3 most similar kernels with measured answers (from other applications)
Reply with ONE JSON object and nothing else, either
  to call a tool:  {"action": "tool", "tool": "fact_sheet" | "junior_opinion" | "similar_kernels"}
  to answer:       {"action": "answer", "device": "CPU" or "GPU", "confidence": number between 0 and 1, "reason": "at most two sentences citing the evidence you used"}
Each tool can be called once; after {max_calls} tool calls you must answer."""


# ----------------------------------------------------------------------------
# junior
# ----------------------------------------------------------------------------
def junior_predictions(df, rounds, seed=JUNIOR_SEED):
    """Random Forest on the hand features, trained per round on study rows only.
    Returns one row per exam row with p_gpu, prediction and confidence."""
    out = []
    for r in rounds:
        study, ex = df.loc[r["study"]], df.loc[r["exam"]]
        rf = RandomForestClassifier(n_estimators=300, random_state=seed, n_jobs=-1)
        rf.fit(F.forest(study), study["label"])
        p = rf.predict_proba(F.forest(ex))[:, 1]
        out.append(pd.DataFrame({"round": r["name"], "row_id": ex["row_id"].to_numpy(),
                                 "label": ex["label"].to_numpy(), "p_gpu": p}))
    j = pd.concat(out, ignore_index=True)
    j["junior_pred"] = (j["p_gpu"] >= 0.5).astype(int)
    j["junior_conf"] = np.maximum(j["p_gpu"], 1 - j["p_gpu"])
    j["hard"] = j["junior_conf"] < THRESHOLD
    return j


# ----------------------------------------------------------------------------
# tools
# ----------------------------------------------------------------------------
def _bytes(n):
    n = float(n)
    for unit in ["bytes", "KB", "MB", "GB"]:
        if n < 1024 or unit == "GB":
            return f"{n:.0f} {unit}" if unit == "bytes" else f"{n:.1f} {unit}"
        n /= 1024


def fact_sheet(row):
    ops = row["comp"] + row["mem"]
    ratio = lambda a, b: (a / b) if b else 0.0
    return "\n".join([
        f"Arithmetic operations: {int(row['comp'])}; comparisons: {int(row['rational'])}",
        f"Global memory accesses: {int(row['mem'])}; local-memory accesses: {int(row['localmem'])}; atomic operations: {int(row['atomic'])}",
        f"Coalesced memory accesses: {int(row['coalesced'])} (counted separately by the static analyser, so it can exceed the global count)",
        f"Arithmetic per global memory access: {ratio(row['comp'], row['mem']):.2f}",
        f"Share of memory accesses to local memory: {100 * ratio(row['localmem'], row['mem'] + row['localmem']):.0f}%",
        f"Data transferred for this launch: {_bytes(row['transfer'])} ({int(row['transfer'])} bytes)",
        f"Work-group size: {int(row['wgsize'])}",
        f"Bytes transferred per counted operation: {ratio(row['transfer'], ops):.1f}",
    ])


def junior_opinion(jrow, n_study):
    dev = "GPU" if jrow["junior_pred"] == 1 else "CPU"
    return (f"A Random Forest trained on {n_study} measured cases from other applications predicts {dev}, "
            f"with probability {jrow['junior_conf']:.2f} (it is only sure above {THRESHOLD:.2f}).")


class SimilarKernels:
    """Nearest study kernels by GraphCodeBERT embedding (cosine similarity).
    Only study rows are searched, so the exam application can never be retrieved."""

    def __init__(self, vectors, k=3):
        self.vectors, self.k = vectors, k

    def __call__(self, row, study):
        q = self.vectors[embed_lib.code_id(row["src"])]
        cand = study.assign(cid=embed_lib.code_ids(study))
        ids = cand["cid"].unique()
        mat = np.stack([self.vectors[i] for i in ids])
        sims = mat @ q / (np.linalg.norm(mat, axis=1) * np.linalg.norm(q) + 1e-12)
        lines = []
        for i in np.argsort(-sims)[: self.k]:
            rows = cand[cand["cid"] == ids[i]]
            near = rows.iloc[np.argsort(np.abs(np.log1p(rows["transfer"]) - np.log1p(row["transfer"])))[:2]]
            cases = "; ".join(
                f"transfer {_bytes(n['transfer'])}, work-group {int(n['wgsize'])}: "
                f"{'GPU' if n['label'] == 1 else 'CPU'} faster "
                f"(CPU/GPU time ratio {n['runtime_cpu'] / n['runtime_gpu']:.2f})"
                for _, n in near.iterrows())
            lines.append(f"- {rows.iloc[0]['benchmark']} (similarity {sims[i]:.2f}): {cases}")
        return "Most similar kernels from other applications, with measured results:\n" + "\n".join(lines)


def kernel_block(row):
    code = devmap.code_key(row["src"])
    if len(code) > CODE_CHARS:
        code = code[:CODE_CHARS] + " ... [truncated]"
    return (f"Kernel code:\n{code}\n\n"
            f"Job size for this launch: transfer {_bytes(row['transfer'])} ({int(row['transfer'])} bytes), "
            f"work-group size {int(row['wgsize'])}.")


# ----------------------------------------------------------------------------
# prompts, parsing
# ----------------------------------------------------------------------------
def base_prompt(machine, row):
    return TASK.format(platform=PLATFORMS[machine]) + "\n\n" + kernel_block(row)


def parse_json(text):
    """First JSON object in the reply, or None."""
    if not text:
        return None
    m = re.search(r"\{.*\}", text, flags=re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None


def parse_answer(obj):
    if not isinstance(obj, dict):
        return None
    dev = str(obj.get("device", "")).strip().upper()
    if dev not in ("CPU", "GPU"):
        return None
    try:
        conf = float(obj.get("confidence", np.nan))
    except (TypeError, ValueError):
        conf = np.nan
    return {"device": dev, "confidence": conf, "reason": str(obj.get("reason", ""))[:500]}


# ----------------------------------------------------------------------------
# one case
# ----------------------------------------------------------------------------
def run_case(llm, variant, machine, row, jrow, study, similar):
    """Returns a dict with the answer, the tools used, tokens and any failure."""
    tools = {
        "fact_sheet": lambda: fact_sheet(row),
        "junior_opinion": lambda: junior_opinion(jrow, len(study)),
        "similar_kernels": lambda: similar(row, study),
    }
    usage = {"calls": 0, "prompt_tokens": 0, "output_tokens": 0}

    def ask(prompt):
        text, u = llm(prompt)
        usage["calls"] += 1
        usage["prompt_tokens"] += int(u.get("prompt_tokens", 0) or 0)
        usage["output_tokens"] += int(u.get("output_tokens", 0) or 0)
        return text

    base = base_prompt(machine, row)
    used = []
    if variant != "agent":
        extra = {"code": [], "facts": ["fact_sheet"],
                 "all_tools": ["fact_sheet", "junior_opinion", "similar_kernels"]}[variant]
        parts = [base] + [f"{name.replace('_', ' ').capitalize()}:\n{tools[name]()}" for name in extra]
        prompt = "\n\n".join(parts + [ANSWER_FORMAT])
        ans = parse_answer(parse_json(ask(prompt)))
        if ans is None:                                  # one retry with a reminder
            ans = parse_answer(parse_json(ask(prompt + "\nYour previous reply was not valid JSON. Reply with the JSON object only.")))
        used = extra
    else:
        transcript = [base, AGENT_FORMAT.replace("{max_calls}", str(MAX_TOOL_CALLS))]
        ans, bad = None, 0
        while ans is None:
            obj = parse_json(ask("\n\n".join(transcript)))
            if isinstance(obj, dict) and obj.get("action") == "tool" and obj.get("tool") in tools \
                    and obj.get("tool") not in used and len(used) < MAX_TOOL_CALLS:
                name = obj["tool"]
                used.append(name)
                transcript.append(f"You called {name}. Result:\n{tools[name]()}")
                continue
            ans = parse_answer(obj)
            if ans is None:
                bad += 1
                if bad > 1:
                    break
                transcript.append("That reply was not valid (unknown or repeated tool, or bad JSON). "
                                  + ("You must answer now." if len(used) >= MAX_TOOL_CALLS else "Call a tool or answer."))
    junior_dev = "GPU" if jrow["junior_pred"] == 1 else "CPU"
    return {
        "device": ans["device"] if ans else junior_dev,
        "llm_ok": ans is not None,
        "confidence": ans["confidence"] if ans else np.nan,
        "reason": ans["reason"] if ans else "",
        "tools_used": "+".join(used),
        **usage,
    }


# ----------------------------------------------------------------------------
# running everything, with resume
# ----------------------------------------------------------------------------
FIELDS = ["machine", "round", "row_id", "variant", "device", "llm_ok", "confidence", "reason",
          "tools_used", "calls", "prompt_tokens", "output_tokens", "seconds", "model"]


class QuotaExhausted(Exception):
    pass


def done_keys(path):
    if not os.path.exists(path):
        return set()
    d = pd.read_csv(path)
    return set(zip(d["machine"], d["row_id"], d["variant"]))


def run_all(llm, model_name, data_dir, emb_path, out_dir, machines=("amd", "nvidia"),
            variants=VARIANTS, limit=None, log=print):
    """Junior on every exam row (saved to junior_<machine>.csv), then each senior
    variant on the hard rows (appended to senior_answers.csv). Resumable."""
    os.makedirs(out_dir, exist_ok=True)
    similar = SimilarKernels(embed_lib.load_embeddings(emb_path))
    path = os.path.join(out_dir, "senior_answers.csv")
    done = done_keys(path)
    new_file = not os.path.exists(path)
    with open(path, "a", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=FIELDS)
        if new_file:
            w.writeheader()
        for machine in machines:
            df = devmap.load(machine, data_dir).set_index("row_id", drop=False)
            rounds = devmap.build_exam(df, EXAM, seed=SPLIT_SEED)
            junior = junior_predictions(df, rounds)
            junior.to_csv(os.path.join(out_dir, f"junior_{machine}.csv"), index=False)
            hard = junior[junior["hard"]]
            log(f"{machine}: junior {100 * (junior.junior_pred == junior.label).mean():.1f}% overall; "
                f"{len(hard)} of {len(junior)} rows go to the senior")
            study_of = {r["name"]: df.loc[r["study"]] for r in rounds}
            for variant in variants:
                todo = [j for _, j in hard.iterrows() if (machine, int(j["row_id"]), variant) not in done]
                if limit:
                    todo = todo[:limit]
                log(f"  {variant}: {len(todo)} rows to do")
                errors_in_a_row = 0
                for k, jrow in enumerate(todo, 1):
                    t0 = time.time()
                    try:
                        res = run_case(llm, variant, machine, df.loc[int(jrow["row_id"])], jrow,
                                       study_of[jrow["round"]], similar)
                        errors_in_a_row = 0
                    except QuotaExhausted:
                        log("  Free-tier limit reached. Everything finished so far is saved; "
                            "run the notebook again later (limits reset daily) and it continues.")
                        return path
                    except Exception as err:             # one bad case: record it, keep the junior's answer
                        errors_in_a_row += 1
                        if errors_in_a_row >= 5:
                            raise RuntimeError(f"5 errors in a row, stopping. Last error: {err}") from err
                        res = {"device": "GPU" if jrow["junior_pred"] == 1 else "CPU", "llm_ok": False,
                               "confidence": np.nan, "reason": f"ERROR: {type(err).__name__}: {str(err)[:200]}",
                               "tools_used": "", "calls": 0, "prompt_tokens": 0, "output_tokens": 0}
                    w.writerow({"machine": machine, "round": jrow["round"], "row_id": int(jrow["row_id"]),
                                "variant": variant, "seconds": round(time.time() - t0, 2),
                                "model": model_name, **res})
                    fh.flush()
                    if k % 20 == 0 or k == len(todo):
                        log(f"    {variant}: {k}/{len(todo)}")
    return path


# ----------------------------------------------------------------------------
# Gemini
# ----------------------------------------------------------------------------
def gemini_llm(api_key, model, pause=4.0, max_wait=600):
    """A function prompt -> (text, usage) that calls Gemini, waits `pause` seconds
    between calls (free-tier friendly) and backs off when rate-limited. Raises
    QuotaExhausted after waiting more than max_wait seconds in total for one call
    (typically the daily free limit); finished rows are already saved."""
    from google import genai
    from google.genai import types
    client = genai.Client(api_key=api_key)
    config = types.GenerateContentConfig(temperature=0.0, response_mime_type="application/json",
                                         max_output_tokens=2048)
    state = {"last": 0.0}

    def call(prompt):
        waited, delay = 0.0, 10.0
        while True:
            gap = time.time() - state["last"]
            if gap < pause:
                time.sleep(pause - gap)
            try:
                state["last"] = time.time()
                resp = client.models.generate_content(model=model, contents=prompt, config=config)
                um = resp.usage_metadata
                usage = {"prompt_tokens": getattr(um, "prompt_token_count", 0) or 0,
                         "output_tokens": (getattr(um, "candidates_token_count", 0) or 0)
                         + (getattr(um, "thoughts_token_count", 0) or 0)}
                return (resp.text or ""), usage
            except Exception as err:            # rate limits and temporary server errors
                code = getattr(err, "code", None)
                if code in (429, 500, 502, 503, 504) or "RESOURCE_EXHAUSTED" in str(err):
                    if waited >= max_wait:
                        raise QuotaExhausted(str(err)[:300])
                    time.sleep(delay)
                    waited += delay
                    delay = min(delay * 2, 120)
                    continue
                raise
    return call

In [ ]:
%%writefile 06_repeats.py
"""
Repeat-run analysis for the most important exam (unseen_app).

  python 06_repeats.py <folder with preds__*.csv>

Every Transformer was trained 3 times (training seeds 1-3) on the SAME piles
(shuffle seed 1). For a fair comparison the Random Forest is also trained 5
times on those same piles here (forest seeds 1-5).

For each pair of models it reports:
  - mean accuracy and spread (std) over runs
  - the 95% range of the accuracy difference from an application-level
    bootstrap, using each row's average correctness over that model's runs
  - McNemar's exact test for each matching pair of runs (run k vs run k)
Writes results/repeats_summary.csv and results/repeats_pairs.csv.
"""
import glob
import itertools
import os
import sys

import numpy as np
import pandas as pd
from scipy.stats import binomtest
from sklearn.ensemble import RandomForestClassifier

import devmap
import features as F

HERE = os.path.dirname(os.path.abspath(__file__))
RESULTS = os.path.join(HERE, "..", "results")
EXAM = "unseen_app"


def mcnemar_p(a, b):
    only_a, only_b = int(((a == 1) & (b == 0)).sum()), int(((a == 0) & (b == 1)).sum())
    return 1.0 if only_a + only_b == 0 else binomtest(only_a, only_a + only_b, 0.5).pvalue


def app_bootstrap(diff, apps, n=2000, seed=0):
    rng = np.random.RandomState(seed)
    g = pd.Series(diff).groupby(np.asarray(apps))
    sums, counts = g.sum().to_numpy(), g.size().to_numpy()
    k = len(sums)
    stats = [sums[p].sum() / counts[p].sum() for p in (rng.randint(0, k, k) for _ in range(n))]
    return np.percentile(stats, [2.5, 97.5])


def transformer_runs(pred_dir, label, machine):
    """{run name: correctness per row (sorted by row_id)}"""
    runs = {}
    for f in sorted(glob.glob(os.path.join(pred_dir, f"preds__{label}__{machine}__{EXAM}__split1__train*.csv"))):
        d = pd.read_csv(f).sort_values("row_id")
        if len(d) == 680:
            runs["train" + f.split("__train")[-1].split(".")[0]] = (d["label"] == d["pred"]).astype(int).to_numpy()
    return runs


def baseline_runs(machine, name, seeds=(1, 2, 3, 4, 5)):
    df = devmap.load(machine).set_index("row_id", drop=False)
    rounds = devmap.build_exam(df, EXAM, seed=1)
    feats = {"forest": F.forest, "job-numbers": F.job_numbers}[name] if name != "majority" else None
    runs = {}
    for s in (seeds if feats is not None else (1,)):
        correct = pd.Series(0, index=df.index)
        for r in rounds:
            study, ex = df.loc[r["study"]], df.loc[r["exam"]]
            if feats is None:
                pred = np.full(len(ex), int(study["label"].mean() >= 0.5))
            else:
                rf = RandomForestClassifier(n_estimators=300, random_state=s, n_jobs=-1)
                rf.fit(feats(study), study["label"])
                pred = rf.predict(feats(ex))
            correct.loc[ex.index] = (pred == ex["label"].to_numpy()).astype(int)
        runs[f"seed{s}"] = correct.sort_index().to_numpy()
    return runs


def main(pred_dir, models=("graphcodebert+fast", "codeberta+fast")):
    summary, pairs = [], []
    for machine in ["amd", "nvidia"]:
        apps = devmap.load(machine).sort_values("row_id")["app"].to_numpy()
        students = {m: transformer_runs(pred_dir, m, machine) for m in models}
        for b in ["forest", "job-numbers", "majority"]:
            students[b] = baseline_runs(machine, b)
        for name, runs in students.items():
            accs = [100 * r.mean() for r in runs.values()]
            summary.append({"machine": machine, "model": name, "runs": len(accs),
                            "mean": round(np.mean(accs), 1), "spread": round(np.std(accs, ddof=1), 1) if len(accs) > 1 else 0.0,
                            "each run": ", ".join(f"{a:.1f}" for a in accs)})
        for a, b in itertools.combinations(students, 2):
            if a not in models:
                continue
            ra, rb = students[a], students[b]
            diff = np.mean(list(ra.values()), axis=0) - np.mean(list(rb.values()), axis=0)
            lo, hi = app_bootstrap(diff, apps)
            ps = [mcnemar_p(x, y) for x, y in zip(ra.values(), itertools.cycle(rb.values()))]
            pairs.append({"machine": machine, "model": a, "vs": b,
                          "difference": round(100 * diff.mean(), 1),
                          "95% range": f"{100 * lo:+.1f} to {100 * hi:+.1f}",
                          "McNemar p per run": ", ".join(f"{p:.3f}" for p in ps),
                          "runs with p<0.05": f"{sum(p < 0.05 for p in ps)}/{len(ps)}"})
    s, p = pd.DataFrame(summary), pd.DataFrame(pairs)
    s.to_csv(os.path.join(RESULTS, "repeats_summary.csv"), index=False)
    p.to_csv(os.path.join(RESULTS, "repeats_pairs.csv"), index=False)
    pd.set_option("display.width", 220)
    print(f"EXAM: {EXAM}, all models on the same piles\n")
    print(s.to_string(index=False))
    print()
    print(p.to_string(index=False))


if __name__ == "__main__":
    main(sys.argv[1])

In [ ]:
%%writefile 09_agent_eval.py
"""
Scores the Phase 3 junior + senior system.

  python 09_agent_eval.py <folder with junior_<machine>.csv and senior_answers.csv>

For each machine and senior variant:
  - accuracy on the hard rows only (junior vs senior)
  - overall accuracy of the combined system (junior on easy rows + senior on hard rows)
  - McNemar's exact test and an application-level bootstrap 95% range for
    "combined system minus junior alone"
  - how often the senior's reply was unusable (junior's answer kept), which tools
    the agent chose, LLM calls and tokens per hard row
Only variants that have answered every hard row are scored (others are reported
as unfinished).
"""
import importlib
import os
import sys

import numpy as np
import pandas as pd

import devmap

rep = importlib.import_module("06_repeats")


def main(folder, data_dir=devmap.DATA_DIR, log=print):
    senior = pd.read_csv(os.path.join(folder, "senior_answers.csv"))
    senior = senior.drop_duplicates(["machine", "row_id", "variant"], keep="last")
    rows, tools = [], []
    for machine in ["amd", "nvidia"]:
        jpath = os.path.join(folder, f"junior_{machine}.csv")
        if not os.path.exists(jpath):
            continue
        j = pd.read_csv(jpath).sort_values("row_id").reset_index(drop=True)
        apps = devmap.load(machine, data_dir).set_index("row_id").loc[j["row_id"], "app"].to_numpy()
        hard = j["hard"].to_numpy()
        junior_ok = (j["junior_pred"] == j["label"]).to_numpy().astype(int)
        rows.append({"machine": machine, "variant": "junior only", "hard rows": int(hard.sum()),
                     "hard-row accuracy": round(100 * junior_ok[hard].mean(), 1),
                     "overall accuracy": round(100 * junior_ok.mean(), 1)})
        for variant, s in senior[senior["machine"] == machine].groupby("variant"):
            s = s.set_index("row_id")
            missing = set(j.loc[hard, "row_id"]) - set(s.index)
            if missing:
                rows.append({"machine": machine, "variant": variant,
                             "note": f"unfinished: {len(missing)} of {int(hard.sum())} hard rows still to do"})
                continue
            pred = j["junior_pred"].to_numpy().copy()
            hard_ids = j.loc[hard, "row_id"].to_numpy()
            pred[hard] = (s.loc[hard_ids, "device"] == "GPU").astype(int).to_numpy()
            ok = (pred == j["label"].to_numpy()).astype(int)
            lo, hi = rep.app_bootstrap(ok - junior_ok, apps)
            sh = s.loc[hard_ids]
            rows.append({
                "machine": machine, "variant": variant, "hard rows": int(hard.sum()),
                "hard-row accuracy": round(100 * ok[hard].mean(), 1),
                "overall accuracy": round(100 * ok.mean(), 1),
                "gain vs junior": round(100 * (ok.mean() - junior_ok.mean()), 1),
                "95% range": f"{100 * lo:+.1f} to {100 * hi:+.1f}",
                "McNemar p": round(rep.mcnemar_p(ok, junior_ok), 4),
                "unusable replies": int((~sh["llm_ok"].astype(bool)).sum()),
                "LLM calls / row": round(sh["calls"].mean(), 2),
                "tokens / row": int((sh["prompt_tokens"] + sh["output_tokens"]).mean()),
            })
            if variant == "agent":
                used = sh["tools_used"].fillna("")
                for name in ["fact_sheet", "junior_opinion", "similar_kernels"]:
                    tools.append({"machine": machine, "tool": name,
                                  "used in % of hard rows": round(100 * used.str.contains(name).mean(), 1)})
                tools.append({"machine": machine, "tool": "(no tool at all)",
                              "used in % of hard rows": round(100 * (used == "").mean(), 1)})
    out = pd.DataFrame(rows)
    out.to_csv(os.path.join(folder, "agent_results.csv"), index=False)
    pd.set_option("display.width", 250)
    log(out.to_string(index=False))
    if tools:
        t = pd.DataFrame(tools)
        t.to_csv(os.path.join(folder, "agent_tool_use.csv"), index=False)
        log("\nWhich tools the agent chose:")
        log(t.to_string(index=False))
    return out


if __name__ == "__main__":
    main(sys.argv[1])

## Settings

In [ ]:
import os
MODEL = "gemini-3.5-flash-lite"     # cheap, fast Gemini model; free tier
PAUSE_SECONDS = 4.0                  # wait between questions (keeps under free-tier per-minute limits)
VARIANTS = ["all_tools", "agent", "facts", "code"]   # most important first
LIMIT = None                         # e.g. 5 to try only 5 rows per variant first

KEY = None
EMB_PATH, OUT_DIR = "embeddings/embeddings__graphcodebert.npz", "phase3"
try:
    from google.colab import userdata, drive
    KEY = userdata.get("GEMINI_API_KEY")
    drive.mount("/content/drive")                     # tick "Select all"
    EMB_PATH = "/content/drive/MyDrive/devmap_results/embeddings/embeddings__graphcodebert.npz"
    OUT_DIR = "/content/drive/MyDrive/devmap_results/phase3"
except ImportError:
    KEY = os.environ.get("GEMINI_API_KEY")
os.makedirs(OUT_DIR, exist_ok=True)
print("Key found:", bool(KEY))
print("GraphCodeBERT embeddings found:", os.path.exists(EMB_PATH))
print("Answers will be saved in:", OUT_DIR)

## Connection check (one tiny question)

In [ ]:
import agent_lib as A
llm = A.gemini_llm(KEY, MODEL, pause=PAUSE_SECONDS)
try:
    reply, usage = llm('Reply with exactly this JSON object: {"ok": true}')
    print("Connection OK. Reply:", reply.strip()[:80], "| tokens:", usage)
except Exception as err:
    print("Connection FAILED:", type(err).__name__, str(err)[:300])
    try:
        from google import genai
        names = [m.name for m in genai.Client(api_key=KEY).models.list() if "flash" in m.name]
        print("Models your key can use (set MODEL to one of these, without 'models/'):", names)
    except Exception as err2:
        print("Could not list models either:", str(err2)[:200])

## Run (resumable)

In [ ]:
import agent_lib as A
path = A.run_all(llm, MODEL, "data", EMB_PATH, OUT_DIR, variants=VARIANTS, limit=LIMIT)
print("\nAnswers saved in", path)

## Results so far

In [ ]:
import importlib
ev = importlib.import_module("09_agent_eval")
results = ev.main(OUT_DIR, data_dir="data")